(Touring-ADR)=
# Touring ADR

As we have already seen in the [surface-bulk case](SurfaceBulk-ADR), nonlinear couplings can be prescribed. We show here a simple example of the development of Turing patters inside a domain with zero boundary flux conditions.

This is to demonstrate that the simulation can be run step by step using its generator structure. Note that the first step is the initialization. This property is used here to modify the initial conditions so to set them as random values.

This property is also used to construct a multidimensional gridfunction used for plotting purposes (although it is a very expensive process, reason why we usually don't do it).


In [2]:
from ngsolve import *
from ngsolve.webgui import Draw
from cosmos.solvers.simulation import Simulation
import time
import numpy as np

mesh = Mesh(unit_square.GenerateMesh(maxh = 0.05))

dt = Parameter(0.1)
T = 5

t = Parameter(0)
simulation = Simulation(mesh=mesh, t=t, dt = dt, T = T)

from cosmos.solvers.adr import ADRSolver
solver = ADRSolver()
simulation.AddSolver(solver)

from cosmos.solvers.tools import gradient

D_A = 0.0001  # diffusion coefficient of species A
d_ratio = 20.0  # ratio between diffusion coefficients
L = 1.0  # typical length scale for nondimensionalization
a_val = 0.1
b_val = 1
Ainit = a_val + b_val
Binit = b_val / (a_val + b_val)**2
gStar = 10000*D_A/L**2
cref = 1

# Solver 1
d1 = D_A
neu_d1 = {'.*': CF((0,0))}
solver.AddSpecie(VorB = VOL,
                u0 = Ainit,
                diffusion = d1,
                reaction = cref*gStar*cref,
                neu_d = neu_d1)

# Solver 1
d2 = D_A*d_ratio
neu_d2 = {'.*': CF((0,0))}
solver.AddSpecie(VorB = VOL,
                u0 = Binit,
                diffusion = d2,
                neu_d = neu_d2)


def f1(trial):
    return -cref*gStar*(a_val + (trial[0]/cref)**2 * (trial[1]/cref))
solver.AddNonlinearity(marker0 = 0, markers = [0, 1], f = f1, VorB = VOL)


def f2(trial):
    return -cref*gStar*b_val + cref*gStar* (trial[0]/cref)**2 * (trial[1]/cref)
solver.AddNonlinearity(marker0 = 1, markers = [0, 1], f = f2, VorB = VOL)


for i, _ in enumerate(simulation()):

    if i == 0:

        gfut1 = GridFunction(solver.gfu_old.components[0].space, multidim = 0)
        gfut2 = GridFunction(solver.gfu_old.components[1].space, multidim = 0)

        solver.gfu_old.components[0].vec.data = Ainit + 0.1*(2*np.random.rand(len(solver.gfu.components[0].vec.data))-1)
        solver.gfu_old.components[1].vec.data = Binit + 0.1*(2*np.random.rand(len(solver.gfu.components[0].vec.data))-1)

    gfut1.AddMultiDimComponent(solver.gfu_old.components[0].vec)
    gfut2.AddMultiDimComponent(solver.gfu_old.components[1].vec)

Draw(gfut1, mesh, interpolate_multidim=True, animate=True)
Draw(gfut2, mesh, interpolate_multidim=True, animate=True)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene